# Portfolio Return Prediction and Construction with Ridge Regression

A rolling-window machine-learning framework for predicting stock returns from financial characteristics and evaluating prediction-based portfolio signals.

> **Academic context.** This notebook is a portfolio adaptation of graduate coursework completed in *Applied Machine Learning in Finance* at the University of Melbourne. It has been reorganised and revised for public presentation. Course materials and raw data are not included.

The analysis is educational and does not constitute investment advice.

## Project Overview

The project examines how Ridge regularisation affects out-of-sample return forecasts and the risk-adjusted performance of prediction-based signal portfolios. Models are estimated using rolling three-year training windows, used to generate predictions for the following 12 months, and retrained annually across 11 values of α, ranging from 10⁻⁵ to 10⁵.

The main empirical distinction is between:

- **realised performance:** predicted-return weights multiplied by subsequent realised returns; and
- **model-implied expected performance:** the same weights multiplied by predicted returns.

This distinction allows the model's ex-ante view to be compared with ex-post outcomes.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## Data & Features

The dataset was provided through the University of Melbourne course environment. The raw data are not included in this repository.

To reproduce the analysis with an authorised copy, create a `raw` directory and place the file at `data/raw/factor_and_ret_oos.csv`.

The notebook expects:

- `id`: firm identifier
- `date`: observation date
- `ret_future`: realised future return used as the prediction target and for out-of-sample portfolio evaluation
- Financial characteristic columns, including:
  - `at_be`: book leverage
  - `ret_6_0`: momentum (0–6 months)
  - `beta_60m`: market beta

In [ ]:
DATA_PATH = Path("data/raw/factor_and_ret_oos.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Raw data are not included. Place an authorised copy at "
        f"{DATA_PATH.resolve()} and rerun the notebook."
    )

df = pd.read_csv(DATA_PATH, index_col=["id", "date"], parse_dates=["date"])
df = df.drop(columns=["Unnamed: 0"], errors="ignore").sort_index()

future_return = df[["ret_future"]].copy()
raw_factors = df.drop(columns="ret_future").copy()

print(f"Observations: {len(df):,}")
print(f"Firm-level characteristics: {raw_factors.shape[1]:,}")
print(f"Date range: {df.index.get_level_values('date').min():%Y-%m-%d} to "
      f"{df.index.get_level_values('date').max():%Y-%m-%d}")

## Rolling Factor Analysis

Book leverage (`at_be`), six-month momentum (`ret_6_0`) and market beta (`beta_60m`) are examined using three-year windows advanced in six-month steps. Missing observations are set to zero for this diagnostic, matching the original analytical convention.

Time variation matters even when pairwise correlations are weak: changing dependence among characteristics can alter diversification and portfolio risk. This motivates rolling estimation rather than assuming one static relationship over the full sample.

In [ ]:
dates = df.index.get_level_values("date")
rolling_window = pd.DateOffset(months=36)
step = pd.DateOffset(months=6)
records = []

for window_end in pd.date_range(dates.min() + rolling_window, dates.max(), freq=step):
    in_window = (dates >= window_end - rolling_window) & (dates < window_end)
    corr = df.loc[in_window, ["at_be", "ret_6_0", "beta_60m"]].fillna(0).corr()
    records.append({
        "date": window_end,
        "Book leverage vs momentum": corr.loc["at_be", "ret_6_0"],
        "Momentum vs market beta": corr.loc["ret_6_0", "beta_60m"],
        "Book leverage vs market beta": corr.loc["at_be", "beta_60m"],
    })

rolling_correlations = pd.DataFrame(records).set_index("date")
rolling_correlations.head()

In [ ]:
ax = rolling_correlations.plot(figsize=(12, 6), linewidth=1.8)
ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax.set(title="Rolling Three-Year Factor Correlations at Six-Month Intervals",
       xlabel="Date", ylabel="Correlation")
plt.tight_layout()
plt.show()

## Data Preprocessing

Characteristics with more than 25% missing observations are removed. Retained features are standardised across the available sample and remaining missing values are replaced with zero, corresponding to the post-standardisation mean.

This preserves the preprocessing convention used in the original analysis. In a production forecasting system, scaling parameters should instead be estimated inside each training window to eliminate any possible look-ahead in preprocessing.

In [ ]:
factors_na_ratio = raw_factors.isna().mean()
factors_to_keep = factors_na_ratio[factors_na_ratio <= 0.25].index

factors = raw_factors.loc[:, factors_to_keep].copy()
factor_std = factors.std().replace(0, np.nan)
factors = ((factors - factors.mean()) / factor_std).fillna(0.0)

print(f"Retained {len(factors_to_keep)} of {raw_factors.shape[1]} characteristics.")

## Ridge Regression

Ridge regression shrinks coefficients through an L2 penalty. The regularisation strength, α, controls the trade-off between fitting cross-sectional return signals and stabilising predictions.

In [ ]:
def predict_ridge(X_train, y_train, X_test, alpha):
    model = Ridge(alpha=alpha, solver="lsqr")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return y_pred

## Rolling Out-of-Sample Prediction

Each model is estimated using the previous 36 months and then used to forecast the following 12 months. Annual retraining respects the time ordering of the data. Predictions are stored alongside the realised forward return for each firm-month.

In [ ]:
log_alphas = np.arange(-5.0, 6.0)
monthly_dates = np.sort(factors.index.get_level_values("date").unique())
rolling_window = 36
retraining_frequency = 12
prediction_blocks = []

for start_idx in range(rolling_window, len(monthly_dates), retraining_frequency):
    train_dates = monthly_dates[start_idx - rolling_window:start_idx]
    test_dates = monthly_dates[start_idx:min(start_idx + retraining_frequency, len(monthly_dates))]

    is_train = factors.index.get_level_values("date").isin(train_dates)
    is_test = factors.index.get_level_values("date").isin(test_dates)
    X_train, y_train = factors.loc[is_train], future_return.loc[is_train, "ret_future"]
    X_test = factors.loc[is_test]
    block = future_return.loc[is_test].copy()

    for log_alpha in log_alphas:
        column = f"logalpha_{log_alpha:.1f}"
        block[column] = predict_ridge(X_train, y_train, X_test, 10 ** log_alpha)

    prediction_blocks.append(block)

df_pred = pd.concat(prediction_blocks).sort_index()
df_pred.head()

## Portfolio Construction

For each firm $i$, the model produces a forecast at time $t$ for the subsequent return:

$$\hat r_{i,t+1\mid t}(\alpha).$$

Following the original analytical setup, the forecast is used directly as an unnormalised portfolio signal:

$$w_{i,t}(\alpha)=\hat r_{i,t+1\mid t}(\alpha).$$

The realised signal-portfolio payoff is:

$$r^{\mathrm{realised}}_{p,t+1}=\sum_i w_{i,t}r_{i,t+1}.$$

These signals are not normalised to sum to one and are not subject to long-only, leverage or other portfolio constraints. The resulting series should therefore be interpreted as unnormalised signal-portfolio payoffs rather than returns from fully invested portfolios.

In [ ]:
prediction_columns = [c for c in df_pred if c.startswith("logalpha_")]

def annualised_sharpe(monthly_returns):
    volatility = monthly_returns.std()
    return np.sqrt(12) * monthly_returns.mean() / volatility if volatility != 0 else np.nan

realised_returns = pd.DataFrame({
    column: (df_pred[column] * df_pred["ret_future"]).groupby("date").sum()
    for column in prediction_columns
})

realised_sharpe = pd.Series(
    {column: annualised_sharpe(realised_returns[column]) for column in prediction_columns},
    name="realised_sharpe",
)

realised_summary = realised_sharpe.rename_axis("prediction").reset_index()
realised_summary["log_alpha"] = realised_summary["prediction"].str.replace("logalpha_", "", regex=False).astype(float)
realised_summary["alpha"] = 10 ** realised_summary["log_alpha"]
realised_summary[["alpha", "realised_sharpe"]].sort_values("alpha")

## Hyperparameter Evaluation

The original analysis produced its highest realised annualised Sharpe ratio of 0.309 at α = 1,000. The table below reports the results across all tested regularisation strengths. Because the raw data are not included in this repository, these values are presented as documented results from the original analysis.

Users with access to the original data can reproduce the analysis by running the preceding cells.

| Alpha | Realised Sharpe |
|---:|---:|
| 0.00001 | 0.059 |
| 0.00010 | 0.059 |
| 0.00100 | 0.059 |
| 0.01000 | 0.059 |
| 0.10000 | 0.057 |
| 1 | 0.073 |
| 10 | 0.103 |
| 100 | 0.126 |
| **1,000** | **0.309** |
| 10,000 | 0.227 |
| 100,000 | 0.108 |

In [ ]:
best_realised = realised_summary.loc[realised_summary["realised_sharpe"].idxmax()]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(realised_summary["log_alpha"], realised_summary["realised_sharpe"], marker="o")
ax.axvline(best_realised["log_alpha"], color="tab:red", linestyle="--", alpha=0.7)
ax.set(title="Realised Sharpe Ratio Across Ridge Penalties",
       xlabel="log10(α)", ylabel="Annualised Sharpe ratio")
plt.tight_layout()
plt.show()

print(f"Best realised alpha: {best_realised['alpha']:g}")
print(f"Best realised Sharpe ratio: {best_realised['realised_sharpe']:.3f}")

In [ ]:
selected_alphas = [0.00001, 1, 100000, best_realised["alpha"]]

for alpha in selected_alphas:
    log_alpha = np.log10(alpha)
    column = f"logalpha_{log_alpha:.1f}"
    realised_return = realised_returns[column]

    plt.figure(figsize=(12, 5))
    plt.plot(realised_return.index, realised_return.values, label=f"alpha={alpha:g}")
    plt.title(f"Portfolio Returns for Alpha={alpha:g}")
    plt.xlabel("Date")
    plt.ylabel("Portfolio Returns")
    plt.legend()
    plt.grid(True)
    plt.show()

## Expected vs Realised Performance

The model-implied expected signal payoff uses the predicted return both as the portfolio signal and as the expected subsequent return:

$$r^{\mathrm{expected}}_{p,t+1\mid t}=\sum_i w_{i,t}\hat r_{i,t+1\mid t}=\sum_i \hat r_{i,t+1\mid t}^{2}.$$

In the original analysis, α = 100,000 produced the highest model-implied expected Sharpe ratio of 2.754, while α = 1,000 produced the highest realised Sharpe ratio of 0.309. This difference shows that the regularisation strength favoured by the model's ex-ante assessment did not deliver the strongest ex-post performance.

For an apples-to-apples comparison, the analysis below fixes α = 100,000—the expected-Sharpe winner—and uses the same prediction-derived signals in both series. The expected payoff is calculated as predicted return × predicted return, while the realised payoff is calculated as the same prediction-derived signal × realised future return.

In [ ]:
expected_returns = pd.DataFrame({
    column: (df_pred[column] * df_pred[column]).groupby("date").sum()
    for column in prediction_columns
})
expected_sharpe = pd.Series(
    {column: annualised_sharpe(expected_returns[column]) for column in prediction_columns},
    name="expected_sharpe",
)

expected_summary = expected_sharpe.rename_axis("prediction").reset_index()
expected_summary["log_alpha"] = expected_summary["prediction"].str.replace("logalpha_", "", regex=False).astype(float)
expected_summary["alpha"] = 10 ** expected_summary["log_alpha"]
display(expected_summary[["alpha", "expected_sharpe"]].sort_values("alpha"))

comparison_alpha_column = expected_sharpe.idxmax()
comparison_log_alpha = float(comparison_alpha_column.replace("logalpha_", ""))
comparison_alpha = 10 ** comparison_log_alpha

expected_same_portfolio = expected_returns[comparison_alpha_column]
realised_same_portfolio = realised_returns[comparison_alpha_column]

comparison = pd.concat(
    [expected_same_portfolio.rename("Expected"), realised_same_portfolio.rename("Realised")],
    axis=1,
)

ax = comparison.plot(figsize=(12, 6), linewidth=1.3, alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title=f"Expected vs Realised Returns for the Same Portfolio (alpha={comparison_alpha:g})",
       xlabel="Date", ylabel="Monthly portfolio return")
plt.tight_layout()
plt.show()

pd.DataFrame({
    "series": ["Expected", "Realised"],
    "alpha": [comparison_alpha, comparison_alpha],
    "annualised_sharpe": [annualised_sharpe(expected_same_portfolio),
                           annualised_sharpe(realised_same_portfolio)],
})

## Key Findings

1. Pairwise factor correlations were generally close to zero, although the momentum–beta relationship showed greater variation over time. This time variation supports the use of rolling analysis rather than assuming constant relationships throughout the sample.
2. Regularisation materially affected signal-portfolio performance. In the original analysis, α = 1,000 produced the highest realised annualised Sharpe ratio of 0.309, while α = 100,000 produced the highest model-implied expected Sharpe ratio of 2.754.
3. The regularisation strength selected using model-implied expected performance differed from the one selected using realised performance, illustrating a gap between ex-ante model assessment and ex-post outcomes.
4. When α = 100,000 is held fixed, the model-implied expected Sharpe ratio is 2.754, whereas the realised Sharpe ratio for the same prediction-derived signals is 0.108.

### Limitations

- The raw data are not included in this repository, so full reproduction requires access to the original dataset.
- The signal portfolios use predicted returns directly as unnormalised signals and do not incorporate transaction costs, turnover limits, leverage constraints or a risk-free rate.
- Feature standardisation is performed using full-sample means and standard deviations, following the original analytical setup. A stricter out-of-sample design would estimate these parameters within each training window.
- Results are historical and sample-specific.